# Metric Depth Estimation — Google Colab Training

Use this notebook if you **don't have a GPU** on your computer.  
Google Colab gives you a **free GPU** for training.

## How to use:
1. Open this file in Google Colab: https://colab.research.google.com/
2. Go to Runtime → Change runtime type → GPU
3. Run each cell in order (Shift+Enter)

## Step 0: Upload project + Install dependencies

In [ ]:
# Upload your project ZIP or clone from your repo
# Option A: Upload ZIP
from google.colab import files
uploaded = files.upload()  # Upload metric_depth_project.zip

import zipfile
with zipfile.ZipFile('metric_depth_project.zip', 'r') as z:
    z.extractall('/content/')

%cd /content/metric_depth_project

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# Verify GPU
import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## Step 1: Camera Calibration

In [ ]:
# Use built-in database (change camera name to yours)
!python step1_camera_calibration/calibrate_camera.py --use_database --camera iphone_13

# See all available cameras:
# !python step1_camera_calibration/calibrate_camera.py --list_cameras

## Step 2: Download Dataset

In [ ]:
# Download NYU Depth V2 (~2.8 GB) and extract all samples
# Colab has ~100 GB disk so space is not an issue here
!python step2_data_collection/download_nyu_dataset.py

## Step 3: Train the Model

In [ ]:
# Train (this takes 1-4 hours depending on GPU)
!python step3_training/train.py --config configs/default.yaml

## Step 4: Test Prediction

In [ ]:
# Upload a test image
from google.colab import files
uploaded = files.upload()  # Upload your test image

test_image = list(uploaded.keys())[0]
print(f'Testing on: {test_image}')

In [ ]:
# Run prediction
!python step4_inference/predict.py --image {test_image} --model outputs/best_model.pth --camera camera_params.json

# Display result
import matplotlib.pyplot as plt
from PIL import Image
import glob

comps = glob.glob('depth_output/*comparison*')
if comps:
    img = Image.open(comps[0])
    plt.figure(figsize=(14, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.title('RGB vs Predicted Depth')
    plt.show()

## Step 5: Evaluate Accuracy

In [ ]:
!python step5_validation/accuracy_metrics.py --model outputs/best_model.pth

## Download Trained Model

In [ ]:
# Download your trained model to use locally
from google.colab import files
files.download('outputs/best_model.pth')